# BC Xenium Breast Cancer Demo

This notebook is a cleaned demo version extracted from the original working notebook.

It is organized around the breast cancer Xenium workflow:

1. Load raw Xenium breast cancer data from this repository
2. Inspect repository-local precomputed UNI / BCAM / ASTER-SC inputs
3. Review imputation outputs starting from the precomputed ASTER-SC results
4. Optionally rerun preprocessing / reconstruction / fusion with the packaged scripts

Recommended environment:

```bash
conda activate repro_st_aster
python -m pip install -r requirements_bc_xenium.txt
```


## Data you need to download first

This repository ships the data directories **empty**. Download and unpack into the
paths below (see `raw_data/bc_xenium/README.md` and
`preprocess_data/bc_xenium/README.md`):

```text
# 1. Raw Xenium counts + coordinates + standardized H&E -> raw_data/bc_xenium/
#    <BC_RAW_URL>
# 2. Precomputed UNI-2 / BCAM / ASTER-SC outputs -> preprocess_data/bc_xenium/
#    <BC_PRECOMPUTED_URL>
```

Item 2 is what this notebook reviews, so it is required to run the cells as written.
To regenerate it from item 1 instead, run the five `scripts/*_bc_xenium_*.sh` in order
(UNI-2 extraction needs the gated weights, or download just the `uni/` subdirectory
from `<BC_UNI_FEATURES_URL>`).

Check your download before running the rest:

In [ ]:
import subprocess, sys

from repro_st_aster.common import find_repo_root

REPO_ROOT = find_repo_root()
subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / 'check_data.py'), 'bc_xenium'])

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import scanpy as sc

from repro_st_aster.common import find_repo_root

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'raw_data' / 'bc_xenium'
PRE_DIR = REPO_ROOT / 'preprocess_data' / 'bc_xenium'
UNI_DIR = PRE_DIR / 'uni'
BCAM_DIR = PRE_DIR / 'bcam_input'
INR_DIR = PRE_DIR / 'inr_output'
BCAM_INR_DIR = PRE_DIR / 'bcam_output'
VIS_DIR = PRE_DIR / 'viz'

print('REPO_ROOT =', REPO_ROOT)
for p in [RAW_DIR, UNI_DIR, BCAM_DIR, INR_DIR, BCAM_INR_DIR, VIS_DIR]:
    print(p, 'exists=', p.exists())


## 1. Raw breast cancer inputs

These are the repository-local raw inputs used by the workflow.


In [ ]:
raw_h5 = RAW_DIR / 'cell_feature_matrix.h5'
cell_coord_csv = RAW_DIR / 'cell_coordinates.csv'
he_image_path = RAW_DIR / 'tissue_standardized_0p5um.jpg'

print(raw_h5)
print(cell_coord_csv)
print(he_image_path)

adata_raw = sc.read_10x_h5(raw_h5)
cell_coords = pd.read_csv(cell_coord_csv)
he_img = Image.open(he_image_path)

print('raw adata shape:', adata_raw.shape)
print('cell_coords shape:', cell_coords.shape)
print('he image size:', he_img.size)
adata_raw


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(he_img)
axes[0].set_title('Standardized H&E image')
axes[0].axis('off')

axes[1].scatter(cell_coords['x_standardized'], cell_coords['y_standardized'], s=0.2, alpha=0.4)
axes[1].set_title('Cell coordinates in standardized space')
axes[1].set_aspect('equal')
axes[1].invert_yaxis()
plt.tight_layout()


## 2. Precomputed UNI features

The repository includes precomputed UNI features so readers do not need the gated UNI weights.


In [ ]:
uni_features = np.load(UNI_DIR / 'superpixel_features.npy', mmap_mode='r')
uni_coords = np.load(UNI_DIR / 'superpixel_coordinates.npz')
with open(UNI_DIR / 'extraction_metadata.json', 'r', encoding='utf-8') as fh:
    uni_meta = json.load(fh)

print('UNI feature map shape:', uni_features.shape)
print('UNI metadata:', uni_meta)
print('UNI coordinate keys:', list(uni_coords.keys()))


In [ ]:
mean_activation = np.asarray(uni_features.mean(axis=2))
std_activation = np.asarray(uni_features.std(axis=2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].imshow(mean_activation, cmap='viridis', aspect='auto')
axes[0].set_title('UNI mean activation')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(std_activation, cmap='plasma', aspect='auto')
axes[1].set_title('UNI feature std')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()


## 3. Precomputed BCAM inputs

These are the repository-local per-cell inputs after mapping cells to UNI superpixels.


In [ ]:
gene_expr = np.load(BCAM_DIR / 'gene_expression_normalized.npy', mmap_mode='r')
uni_per_cell = np.load(BCAM_DIR / 'uni2_features_per_cell.npy', mmap_mode='r')
coords = np.load(BCAM_DIR / 'cell_coords_standardized.npy')
gene_names = np.load(BCAM_DIR / 'gene_names.npy', allow_pickle=True)

print('gene_expr:', gene_expr.shape)
print('uni_per_cell:', uni_per_cell.shape)
print('coords:', coords.shape)
print('n_genes:', len(gene_names))
print('example genes:', gene_names[:10])


## 4. Imputation starts here: precomputed ASTER-SC outputs

The notebook starts the main demonstration from the precomputed imputation outputs, so it is immediately usable without a long retraining step.


In [ ]:
inr_expr = np.load(INR_DIR / 'inr_reconstructed_expression.npy', mmap_mode='r')
fusion_latent = np.load(BCAM_INR_DIR / 'fusion_latent_512.npy', mmap_mode='r')
fusion_expr = np.load(BCAM_INR_DIR / 'fusion_predicted_expression.npy', mmap_mode='r')
labels = np.load(VIS_DIR / 'labels_fusion_K17.npy')

print('INR expression:', inr_expr.shape)
print('Fusion latent:', fusion_latent.shape)
print('Fusion expression:', fusion_expr.shape)
print('Cluster labels:', labels.shape, 'unique=', len(np.unique(labels)))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(coords[:, 0], coords[:, 1], c=labels, s=0.4, cmap='tab20', alpha=0.8)
axes[0].set_title('ASTER-SC / BCAM clustering')
axes[0].set_aspect('equal')
axes[0].axis('off')

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
latent_pca = pca.fit_transform(np.asarray(fusion_latent))
axes[1].scatter(latent_pca[:, 0], latent_pca[:, 1], c=labels, s=0.4, cmap='tab20', alpha=0.5)
axes[1].set_title('Fusion latent PCA')

cluster_ids, cluster_counts = np.unique(labels, return_counts=True)
axes[2].bar(cluster_ids, cluster_counts, color=plt.cm.tab20(cluster_ids / max(cluster_ids.max(), 1)))
axes[2].set_title('Cluster sizes')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Cells')

plt.tight_layout()


In [ ]:
marker_genes = ['KRT14', 'EPCAM', 'CD68', 'COL1A1', 'PECAM1', 'PTPRC']
found = [g for g in gene_names if str(g).upper() in marker_genes]
if not found:
    found = list(gene_names[:6])
found = found[:6]
print('markers used:', found)

fig, axes = plt.subplots(len(found), 2, figsize=(10, 4 * len(found)))
if len(found) == 1:
    axes = np.array([axes])

for i, gene in enumerate(found):
    gi = np.where(gene_names == gene)[0][0]
    axes[i, 0].scatter(coords[:, 0], coords[:, 1], c=np.asarray(inr_expr[:, gi]), s=0.3, cmap='turbo', alpha=0.8)
    axes[i, 0].set_title(f'{gene} - INR')
    axes[i, 0].set_aspect('equal')
    axes[i, 0].axis('off')

    axes[i, 1].scatter(coords[:, 0], coords[:, 1], c=np.asarray(fusion_expr[:, gi]), s=0.3, cmap='turbo', alpha=0.8)
    axes[i, 1].set_title(f'{gene} - BCAM fusion')
    axes[i, 1].set_aspect('equal')
    axes[i, 1].axis('off')

plt.tight_layout()


In [ ]:
cluster_mean = []
for cid in np.unique(labels):
    mask = labels == cid
    cluster_mean.append(np.asarray(fusion_expr[mask]).mean(axis=0))
cluster_mean = np.asarray(cluster_mean)

var_idx = np.argsort(cluster_mean.var(axis=0))[-30:]
heat = cluster_mean[:, var_idx]

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(heat.T, aspect='auto', cmap='RdBu_r')
ax.set_title('Top variable genes across clusters')
ax.set_xlabel('Cluster')
ax.set_ylabel('Gene')
ax.set_xticks(range(cluster_mean.shape[0]))
ax.set_yticks(range(len(var_idx)))
ax.set_yticklabels(gene_names[var_idx], fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()


## 5. Optional rerun commands

These cells are not meant for immediate execution in a lightweight demo session. They show how to rerun the packaged scripts from the repository root.


In [ ]:
# !bash scripts/prepare_bc_xenium_uni.sh --model-dir /absolute/path/to/uni2-h
# !bash scripts/prepare_bc_xenium_bcam_input.sh
# !bash scripts/reproduce_bc_xenium_inr.sh
# !bash scripts/reproduce_bc_xenium_bcam.sh
# !bash scripts/reproduce_bc_xenium_cluster.sh
